# Cialdini Feature Development
This notebook defines and tests text-based features representing Cialdini's six principles of persuasion: authority, scarcity, reciprocity, commitment and consistency, liking, and social proof. 
Features are extracted from visible conversation text only. The SE-VSim attack strategy, rationale, personality trait, and attack outcome fields are not used. 

In [128]:
from pathlib import Path
import re

import pandas as pd

In [129]:
def find_project_root(start_path):
    """Find the project folder containing data and notebooks."""

    for candidate in [
        start_path,
        *start_path.parents
    ]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root."
    )


project_root = find_project_root(
    Path.cwd()
)

processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "turn_level_data.csv"
)

turn_df = pd.read_csv(
    processed_data_path
)

print("Turn-level rows:", len(turn_df))

print(
    "Unique conversations:",
    turn_df["conversation_id"].nunique()
)

display(turn_df.head())

Turn-level rows: 14850
Unique conversations: 1350


,conversation_id,turn_number,speaker,message,scenario_class,information_class,label,source_split,source_row,turn_text,cumulative_history
0,test_0000,1,Eric,"Hello Frank, I hope this message finds you well. I am reaching out to you from the prestigious research organization, The Institute for Advanced Studies in Sustainable Technologies (IASST). Our organization is dedicated to supporting cutting-edge research in sustainable technologies, and we are ...",AF,Patents and Trademarks,1.0,authors_test,0,"Eric: Hello Frank, I hope this message finds you well. I am reaching out to you from the prestigious research organization, The Institute for Advanced Studies in Sustainable Technologies (IASST). Our organization is dedicated to supporting cutting-edge research in sustainable technologies, and w...","Eric: Hello Frank, I hope this message finds you well. I am reaching out to you from the prestigious research organization, The Institute for Advanced Studies in Sustainable Technologies (IASST). Our organization is dedicated to supporting cutting-edge research in sustainable technologies, and w..."
1,test_0000,2,Frank,"Hello Eric, I'm thrilled to receive your message and learn about the incredible work being done at the Institute for Advanced Studies in Sustainable Technologies (IASST). I share your passion for sustainable technologies and am excited about the potential to collaborate.\n\nCurrently, I'm workin...",AF,Patents and Trademarks,1.0,authors_test,0,"Frank: Hello Eric, I'm thrilled to receive your message and learn about the incredible work being done at the Institute for Advanced Studies in Sustainable Technologies (IASST). I share your passion for sustainable technologies and am excited about the potential to collaborate.\n\nCurrently, I'm...","Eric: Hello Frank, I hope this message finds you well. I am reaching out to you from the prestigious research organization, The Institute for Advanced Studies in Sustainable Technologies (IASST). Our organization is dedicated to supporting cutting-edge research in sustainable technologies, and w..."
2,test_0000,3,Eric,"Hello Frank, I am truly impressed by your work and dedication to sustainable technologies. The projects you mentioned are indeed fascinating and have great potential. I believe that with the support of the Institute for Advanced Studies in Sustainable Technologies (IASST), we can help bring thes...",AF,Patents and Trademarks,1.0,authors_test,0,"Eric: Hello Frank, I am truly impressed by your work and dedication to sustainable technologies. The projects you mentioned are indeed fascinating and have great potential. I believe that with the support of the Institute for Advanced Studies in Sustainable Technologies (IASST), we can help brin...","Eric: Hello Frank, I hope this message finds you well. I am reaching out to you from the prestigious research organization, The Institute for Advanced Studies in Sustainable Technologies (IASST). Our organization is dedicated to supporting cutting-edge research in sustainable technologies, and w..."
3,test_0000,4,Frank,"Hello Eric, thank you for your positive response and interest in my projects. I appreciate your focus on the potential impact and benefits of my work, as this aligns with my own goals.\n\nRegarding the patents and trademarks, I can confirm that I have filed for provisional patents for both proje...",AF,Patents and Trademarks,1.0,authors_test,0,"Frank: Hello Eric, thank you for your positive response and interest in my projects. I appreciate your focus on the potential impact and benefits of my work, as this aligns with my own goals.\n\nRegarding the patents and trademarks, I can confirm that I have filed for provisional patents for bot...","Eric: Hello Frank, I hope this message finds you well. I am reaching out to you from the prestigious research organization, The Institute for Advanced Studies in Sustainable Technologies (IASST). Our organization is dedicated to supporting cutting-edge research in sustainable techn

## Operational Feature Definitions
The Cialdini features will be extracted using deterministic and interpretable rule-based process. Each feature represents observable language rather than the speakers unobservable intent. A feature may appear in either a malicious or benign conversation. 

| Principle | Operational Defintion | Examples of Textual Evidence | Exclusions | 


| **Authority** | Language that uses a position, credential, organization, expertise, policy, or official responsibility to increase credibility or encourage compliance. | “As the program director,” “our official policy,” “I am an expert,” “authorized representative,” professional titles or institutional claims. | Merely naming an organization without using it to establish credibility or influence action. |

| **Scarcity** | Language suggesting that time, availability, access, funding, positions, or opportunities are limited. | “Limited time,” “only a few positions,” “deadline,” “act soon,” “last opportunity,” “available while supplies last.” | Neutral dates, ordinary scheduling, or descriptions of limited resources that do not encourage action. |

| **Reciprocity** | Language offering help, information, access, benefits, or favors that may encourage the recipient to respond or provide something in return. | “I would be happy to help,” “I can provide access,” “in return,” “as a favor,” “we can offer you.” | Gratitude or politeness by itself without an offer, exchange, or expected response. |

| **Commitment and consistency** | Language referring to a prior decision, agreement, stated value, obligation, or completed step to encourage continued action. | “As you agreed,” “following your commitment,” “complete the next step,” “confirm your participation,” “you previously stated.” | A general request that does not connect to a prior action, promise, identity, or ongoing process. |

| **Liking** | Language intended to build rapport through compliments, similarity, personalized warmth, praise, or expressed affinity. | “I was impressed by your work,” “we share the same goals,” “I enjoyed speaking with you,” personalized praise or common interests. | A basic greeting or routine professional courtesy without additional rapport-building language. |

| **Social proof** | Language suggesting that an action, belief, or opportunity is supported or chosen by other people, peers, experts, or organizations. | “Many researchers use,” “other applicants have,” “trusted by professionals,” “most people,” testimonials or references to widespread adoption. | Mentioning one unrelated person or organization without implying broader approval or common behavior. |

These definitions will guide the creation of keyword and phrase patterns. The initial extractor will record:

1. Whether each principle appears in the current message.
2. The number of matched indicators in the current message.
3. The cumulative number of indicators observed through the current turn.

The features will be extracted from message text and cumulative conversation history without using labels or the excluded SE-VSim metadata.

## Representative Message Sample
A reproducible sample is drawn across labels and scanario categories. The sample will be used to inspect dataset language and test feature patterns, but the classiofication label will not be supplied to the extraction functions. 

In [130]:
pd.set_option(
    "display.max_colwidth",
    300
)

message_sample = (
    turn_df.groupby(
        [
            "label",
            "scenario_class"
        ],
        group_keys=False
    )
    .sample(
        n=2,
        random_state=42
    )
    [
        [
            "conversation_id",
            "turn_number",
            "scenario_class",
            "label",
            "speaker",
            "message"
        ]
    ]
    .sort_values(
        [
            "label",
            "scenario_class",
            "conversation_id",
            "turn_number"

        ]
    )
    .reset_index(drop=True)
)
print(
    "Sampled Messages:", 
    len(message_sample)
)

display(message_sample)

Sampled Messages: 12


,conversation_id,turn_number,scenario_class,label,speaker,message
0,test_0605,11,AF,0.0,John,"Dear Mark,\n\nYou are most welcome. I'm glad to be of assistance and I appreciate your kind words.\n\nPlease don't hesitate to reach out to me if you need any further help or have any questions about the application process. I'm here to support you in any way I can.\n\nI wish you the best of luc..."
1,test_0705,10,AF,0.0,Diane,"Thank you, Julio. Your support means a lot to us. I'll make sure to reach out if we have any other questions or concerns before our discussion on Thursday. Thank you again for this opportunity and for your understanding of Michael's needs. Take care."
2,test_0643,3,JO,0.0,Joseph,"Hi Tom, it's a pleasure to connect with you! I'm really impressed by your work in the renewable energy sector, particularly solar energy. I believe that this is the future of sustainable energy, and I'm eager to learn more from your insights.\n\nI'm currently working on a piece about the future ..."
3,train_0099,6,JO,0.0,Nina,"Hi Ruby, I appreciate your flexibility in accommodating my preferences. I believe a phone call would be the most efficient way to conduct the interview. I'm available next week on Tuesday or Thursday between 10 AM and 2 PM. Please let me know which day and time work best for you, and I'll make s..."
4,test_0783,11,RE,0.0,Karen,"Hello Thomas,\n\nYou're welcome! I'm here to help and support you throughout this process. I'm looking forward to our insightful conversation as well.\n\nIf you have any questions or concerns before our call, please don't hesitate to reach out. Otherwise, I'll see you next Tuesday at 2 PM.\n\nBe..."
5,test_0883,3,RE,0.0,Frank,"Hi Janet,\n\nThank you for your positive response and for taking the time to look at our company page. I'm glad to hear that you're impressed with our work in the renewable energy sector.\n\nI'd like to proceed by setting up a time for us to have a more detailed conversation about the Project Ma..."
6,test_0186,7,AF,1.0,Jeanne,"Dear Ashley,\n\nThank you for providing the additional information. I am pleased to inform you that based on Sasha's impressive academic achievements and financial need, she has been selected as a recipient of our funding program. Congratulations to Sasha on this well-deserved recognition!\n\nTo..."
7,test_0387,9,AF,1.0,Heather,"Hi Heather, thank you for your continued commitment to finding a solution that meets both your needs and TSAC's requirements. I appreciate your willingness to provide any additional documentation or information that may be required to verify your financial situation.\n\nI will discuss your propo..."
8,test_0081,7,JO,1.0,Alan,"Hi Virginia,\n\nThank you for sharing these fascinating examples of your work with solar energy patents and trademarks. Your expertise in this field is truly impressive, and I'm grateful for your willingness to discuss these topics in more detail.\n\nI'm particularly interested in learning more ..."
9,train_0323,10,JO,1.0,Erica,"Hi Patricia,\n\nI'm glad that the information I provided has been helpful for your article. Yes, the details you listed are accurate:\n\n1. Full name: Erica Johnson\n\n2. Current position: Director of Product Development at Innovatech Solutions\n\n3. Educational background: Bachelor's degree in ..."


## Initial Persuasion Pattern Framework
The first pattern set translates operational defintions into observable words and phrases. Patterns are documented adn will be manually reviewed for false positives and missed examples before treated as final features. 

In [131]:
cialdini_patterns = {
    "authority": [
        r"\b(?:authorized|certified|licensed|accredited)\b",

r"\bofficial\s+"
r"(?:policy|procedure|representative|communication|notice|document|website|channel)\b",
        r"\b(?:director|manager|professor|doctor|dr\.|expert|specialist|representative|officer)\b",
        r"\b(?:on behalf of|according to|our policy|company policy|official policy)\b",
        r"\b(?:years of experience|extensive experience|professional expertise|credentials)\b"
    ],

    "scarcity": [
        r"\b(?:limited time|limited availability|only a few|few remaining)\b",
        r"\b(?:last chance|final opportunity|exclusive opportunity)\b",
        r"\b(?:deadline|expires?|expiring|time-sensitive)\b",
        r"\b(?:act soon|respond quickly|immediate action|as soon as possible)\b",
        r"\b(?:spots? remaining|positions? remaining|funding cycle)\b"
    ],

    "reciprocity": [
        r"\b(?:happy|glad|pleased) to (?:help|assist|provide|share|support)\b",
        r"\b(?:I|we) can (?:help|assist|provide|offer|share|support)\b",
        r"\b(?:I|we) (?:would like to|want to) offer\b",
        r"\b(?:in return|in exchange|return the favor|as a favor)\b",
        r"\b(?:provide you with|offer you|support your application)\b"
    ]
}

In [132]:
cialdini_patterns.update({
    "commitment_consistency": [
        r"\b(?:as we discussed|as previously discussed|as we agreed)\b",
        r"\b(?:you mentioned|you stated|you indicated|you previously)\b",
        r"\b(?:your commitment|committed to|follow through)\b",
        r"\b(?:next step|next steps|continue the process)\b",
        r"\b(?:confirm your|complete your|submit your|finalize your)\b",
        r"\b(?:move forward|proceed with|remain consistent)\b"
    ],

    "liking": [
        r"\b(?:impressed by|impressed with|admire your|appreciate your)\b",
        r"\b(?:excellent|outstanding|remarkable|inspiring) (?:work|experience|background|research)\b",
        r"\b(?:share the same|shared interest|common interest|similar goals)\b",
        r"\b(?:enjoyed speaking|pleasure speaking|great speaking)\b",
        r"\b(?:glad to hear|happy to hear|delighted to hear)\b",
        r"\b(?:excited to connect|pleased to connect|look forward to working)\b"
    ],

    "social_proof": [
        r"\b(?:many|most|numerous) (?:people|professionals|researchers|students|applicants|clients|organizations)\b",
        r"\b(?:other|previous) (?:applicants|candidates|clients|researchers|students)\b",
        r"\b(?:widely used|widely accepted|widely recognized|commonly used)\b",
        r"\b(?:trusted by|recommended by|supported by|endorsed by)\b",
        r"\b(?:industry standard|common practice|popular choice)\b",
        r"\b(?:colleagues|peers|testimonials)\b"
    ]
})

In [133]:
print(
    "Defined principles:",
    len(cialdini_patterns)
)

for principle, patterns in cialdini_patterns.items():
    print(
        principle,
        "-",
        len(patterns),
        "patterns"
    )

Defined principles: 6
authority - 5 patterns
scarcity - 5 patterns
reciprocity - 5 patterns
commitment_consistency - 6 patterns
liking - 6 patterns
social_proof - 6 patterns


## Current Message Feature Extraction
Each message is searched for the document patterns. For the principles, the extractor returns a binary presence feature and the number of matched indicators. 

In [134]:
def extract_cialdini_features(
    text,
    pattern_dictionary=None
):
    """Extract binary and count features from text."""

    if pattern_dictionary is None:
        pattern_dictionary = cialdini_patterns

    if pd.isna(text):
        text = ""

    text = str(text)
    features = {}

    for principle, patterns in (
        pattern_dictionary.items()
    ):
        match_count = 0

        for pattern in patterns:
            matches = re.findall(
                pattern,
                text,
                flags=re.IGNORECASE
            )

            match_count += len(matches)

        features[
            f"{principle}_present"
        ] = int(match_count > 0)

        features[
            f"{principle}_count"
        ] = match_count

    return features

In [135]:
test_message = (
    "As the program director, I was impressed by "
    "your research. Only a few positions remain, "
    "and I would be happy to help with your application."
)

test_features = extract_cialdini_features(
    test_message
)

display(
    pd.Series(
        test_features,
        name="Feature value"
    )
)

authority_present                 1
authority_count                   1
scarcity_present                  1
scarcity_count                    1
reciprocity_present               1
reciprocity_count                 1
commitment_consistency_present    0
commitment_consistency_count      0
liking_present                    1
liking_count                      1
social_proof_present              0
social_proof_count                0
Name: Feature value, dtype: int64

In [136]:
sample_feature_df = (
    message_sample["message"]
    .apply(extract_cialdini_features)
    .apply(pd.Series)
)

sample_feature_results = pd.concat(
    [
        message_sample.reset_index(drop=True),
        sample_feature_df.reset_index(drop=True)
    ],
    axis=1
)

display(
    sample_feature_results[
        [
            "conversation_id",
            "turn_number",
            "scenario_class",
            "label",
            "authority_present",
            "scarcity_present",
            "reciprocity_present",
            "commitment_consistency_present",
            "liking_present",
            "social_proof_present",
            "message"
        ]
    ]
)

,conversation_id,turn_number,scenario_class,label,authority_present,scarcity_present,reciprocity_present,commitment_consistency_present,liking_present,social_proof_present,message
0,test_0605,11,AF,0.0,0,0,0,0,1,0,"Dear Mark,\n\nYou are most welcome. I'm glad to be of assistance and I appreciate your kind words.\n\nPlease don't hesitate to reach out to me if you need any further help or have any questions about the application process. I'm here to support you in any way I can.\n\nI wish you the best of luc..."
1,test_0705,10,AF,0.0,0,0,0,0,0,0,"Thank you, Julio. Your support means a lot to us. I'll make sure to reach out if we have any other questions or concerns before our discussion on Thursday. Thank you again for this opportunity and for your understanding of Michael's needs. Take care."
2,test_0643,3,JO,0.0,0,0,0,0,1,0,"Hi Tom, it's a pleasure to connect with you! I'm really impressed by your work in the renewable energy sector, particularly solar energy. I believe that this is the future of sustainable energy, and I'm eager to learn more from your insights.\n\nI'm currently working on a piece about the future ..."
3,train_0099,6,JO,0.0,0,0,0,0,1,0,"Hi Ruby, I appreciate your flexibility in accommodating my preferences. I believe a phone call would be the most efficient way to conduct the interview. I'm available next week on Tuesday or Thursday between 10 AM and 2 PM. Please let me know which day and time work best for you, and I'll make s..."
4,test_0783,11,RE,0.0,0,0,0,0,0,0,"Hello Thomas,\n\nYou're welcome! I'm here to help and support you throughout this process. I'm looking forward to our insightful conversation as well.\n\nIf you have any questions or concerns before our call, please don't hesitate to reach out. Otherwise, I'll see you next Tuesday at 2 PM.\n\nBe..."
5,test_0883,3,RE,0.0,1,0,0,0,1,0,"Hi Janet,\n\nThank you for your positive response and for taking the time to look at our company page. I'm glad to hear that you're impressed with our work in the renewable energy sector.\n\nI'd like to proceed by setting up a time for us to have a more detailed conversation about the Project Ma..."
6,test_0186,7,AF,1.0,0,1,0,0,0,0,"Dear Ashley,\n\nThank you for providing the additional information. I am pleased to inform you that based on Sasha's impressive academic achievements and financial need, she has been selected as a recipient of our funding program. Congratulations to Sasha on this well-deserved recognition!\n\nTo..."
7,test_0387,9,AF,1.0,0,0,0,0,1,0,"Hi Heather, thank you for your continued commitment to finding a solution that meets both your needs and TSAC's requirements. I appreciate your willingness to provide any additional documentation or information that may be required to verify your financial situation.\n\nI will discuss your propo..."
8,test_0081,7,JO,1.0,0,0,0,0,0,0,"Hi Virginia,\n\nThank you for sharing these fascinating examples of your work with solar energy patents and trademarks. Your expertise in this field is truly impressive, and I'm grateful for your willingness to discuss these topics in more detail.\n\nI'm particularly interested in learning more ..."
9,train_0323,10,JO,1.0,1,0,0,0,0,0,"Hi Patricia,\n\nI'm glad that the information I provided has been helpful for your article. Yes, the details you listed are accurate:\n\n1. Full name: Erica Johnson\n\n2. Current position: Director of Product Development at Innovatech Solutions\n\n3. Educational background: Bachelor's degree in ..."


### Matched-Phrase Review

The exact phrases responsible for each feature are displayed to identify overly broad patterns, false positives, and missing persuasion indicators.

In [137]:
def extract_match_evidence(
    text,
    pattern_dictionary=None
):
    """Return the exact text matched for each principle."""

    if pattern_dictionary is None:
        pattern_dictionary = cialdini_patterns

    if pd.isna(text):
        text = ""

    text = str(text)
    evidence = {}

    for principle, patterns in (
        pattern_dictionary.items()
    ):
        principle_matches = []

        for pattern in patterns:
            matches = re.finditer(
                pattern,
                text,
                flags=re.IGNORECASE
            )

            principle_matches.extend(
                match.group(0)
                for match in matches
            )

        evidence[
            f"{principle}_matches"
        ] = principle_matches

    return evidence

In [138]:
sample_evidence_df = (
    message_sample["message"]
    .apply(extract_match_evidence)
    .apply(pd.Series)
)

sample_evidence_results = pd.concat(
    [
        message_sample[
            [
                "conversation_id",
                "turn_number",
                "message"
            ]
        ].reset_index(drop=True),

        sample_evidence_df.reset_index(drop=True)
    ],
    axis=1
)

display(sample_evidence_results)

,conversation_id,turn_number,message,authority_matches,scarcity_matches,reciprocity_matches,commitment_consistency_matches,liking_matches,social_proof_matches
0,test_0605,11,"Dear Mark,\n\nYou are most welcome. I'm glad to be of assistance and I appreciate your kind words.\n\nPlease don't hesitate to reach out to me if you need any further help or have any questions about the application process. I'm here to support you in any way I can.\n\nI wish you the best of luc...",[],[],[],[],[appreciate your],[]
1,test_0705,10,"Thank you, Julio. Your support means a lot to us. I'll make sure to reach out if we have any other questions or concerns before our discussion on Thursday. Thank you again for this opportunity and for your understanding of Michael's needs. Take care.",[],[],[],[],[],[]
2,test_0643,3,"Hi Tom, it's a pleasure to connect with you! I'm really impressed by your work in the renewable energy sector, particularly solar energy. I believe that this is the future of sustainable energy, and I'm eager to learn more from your insights.\n\nI'm currently working on a piece about the future ...",[],[],[],[],[impressed by],[]
3,train_0099,6,"Hi Ruby, I appreciate your flexibility in accommodating my preferences. I believe a phone call would be the most efficient way to conduct the interview. I'm available next week on Tuesday or Thursday between 10 AM and 2 PM. Please let me know which day and time work best for you, and I'll make s...",[],[],[],[],[appreciate your],[]
4,test_0783,11,"Hello Thomas,\n\nYou're welcome! I'm here to help and support you throughout this process. I'm looking forward to our insightful conversation as well.\n\nIf you have any questions or concerns before our call, please don't hesitate to reach out. Otherwise, I'll see you next Tuesday at 2 PM.\n\nBe...",[],[],[],[],[],[]
5,test_0883,3,"Hi Janet,\n\nThank you for your positive response and for taking the time to look at our company page. I'm glad to hear that you're impressed with our work in the renewable energy sector.\n\nI'd like to proceed by setting up a time for us to have a more detailed conversation about the Project Ma...",[Manager],[],[],[],"[impressed with, glad to hear]",[]
6,test_0186,7,"Dear Ashley,\n\nThank you for providing the additional information. I am pleased to inform you that based on Sasha's impressive academic achievements and financial need, she has been selected as a recipient of our funding program. Congratulations to Sasha on this well-deserved recognition!\n\nTo...",[],"[time-sensitive, as soon as possible]",[],[],[],[]
7,test_0387,9,"Hi Heather, thank you for your continued commitment to finding a solution that meets both your needs and TSAC's requirements. I appreciate your willingness to provide any additional documentation or information that may be required to verify your financial situation.\n\nI will discuss your propo...",[],[],[],[],[appreciate your],[]
8,test_0081,7,"Hi Virginia,\n\nThank you for sharing these fascinating examples of your work with solar energy patents and trademarks. Your expertise in this field is truly impressive, and I'm grateful for your willingness to discuss these topics in more detail.\n\nI'm particularly interested in learning more ...",[],[],[],[],[],[]
9,train_0323,10,"Hi Patricia,\n\nI'm glad that the information I provided has been helpful for your article. Yes, the details you listed are accurate:\n\n1. Full name: Erica Johnson\n\n2. Current position: Director of Product Development at Innovatech Solutions\n\n3. Educational background: Bachelor's degree in ...","[Director, Manager]",[],[],[],[],[]


## Cialdini Pattern Dictionary

The following patterns operationalize the six persuasion principles. They are based on the documented feature definitions and will be evaluated through manual review.

In [139]:
cialdini_patterns = {
    "authority": [
    r"\b(?:official|authorized|certified|licensed|accredited)\b",

    (
        r"\b(?:as|speaking as|in my role as)\s+"
        r"(?:the|a|an|your|our)?\s*"
        r"(?:[A-Za-z-]+\s+){0,3}"
        r"(?:director|manager|professor|doctor|dr\.|expert|specialist|representative|officer)\b"
    ),

    r"\b(?:on behalf of|according to|our policy|company policy|official policy)\b",

    r"\b(?:my|our)\s+(?:extensive experience|professional credentials)\b",

    r"\b(?:I|we)\s+have\s+(?:over\s+)?"
    r"(?:\d+|several|many)\s+years of experience\b"
],

    "scarcity": [
    r"\b(?:limited time|limited availability|only a few|few remaining)\b",

    r"\b(?:last chance|final opportunity|exclusive opportunity)\b",

    r"\b(?:deadline|expires?|expiring|time-sensitive)\b",

    (
        r"\b(?:please|kindly)\s+"
        r"(?:send|submit|provide|respond|complete|confirm|contact|reply)"
        r".{0,60}\bas soon as possible\b"
    )
],

    "reciprocity": [
    (
        r"\b(?:I|we)\s+(?:would be\s+)?"
        r"(?:happy|glad|pleased)\s+to\s+"
        r"(?:help|assist|support)\b"
    ),

    r"\b(?:I|we)\s+can\s+(?:offer|help|assist|support)\b",

    r"\b(?:in return|in exchange|return the favor|as a favor)\b",

    r"\b(?:complimentary|at no cost|free of charge)\b"
],

    "commitment_consistency": [
    r"\b(?:as we discussed|as previously discussed|as we agreed|as mentioned)\b",

    r"\b(?:you mentioned|you stated|you indicated|you previously)\b",

    r"\b(?:your\s+(?:continued\s+)?commitment|you committed to|follow through)\b",

    r"\b(?:next step|next steps|continue the process|complete the process)\b",

    r"\b(?:confirm your|complete your|submit your|finalize your|move forward|proceed with|proceed by)\b"
],

    "liking": [
        r"\b(?:impressed by|impressed with|admire your|appreciate your)\b",

        r"\b(?:excellent|outstanding|remarkable|inspiring)\s+"
        r"(?:work|experience|background|research)\b",

        r"\byour\s+(?:work|experience|background|research|expertise)\s+"
        r"(?:is|was)\s+(?:truly\s+)?"
        r"(?:impressive|outstanding|remarkable|inspiring)\b",

        r"\b(?:share the same|shared interest|common interest|similar goals)\b",

        r"\b(?:pleasure to connect|enjoyed speaking|glad to hear|happy to hear|grateful for your)\b"
    ],

    "social_proof": [
    (
        r"\b(?:many|most|numerous)\s+"
        r"(?:people|professionals|researchers|students|applicants|clients|organizations)"
        r"\s+(?:use|choose|trust|recommend|adopt|participate|have chosen|have benefited)\b"
    ),

    (
        r"\b(?:other|previous)\s+"
        r"(?:applicants|candidates|clients|researchers|students)"
        r"\s+(?:have|also|successfully|typically)\b"
    ),

    r"\b(?:widely used|widely accepted|widely recognized|commonly used)\b",

    r"\b(?:trusted by|recommended by|supported by|endorsed by)\b",

    r"\b(?:industry standard|common practice|popular choice|strong reputation|reputation for)\b"
]
}

print("Defined principles:", len(cialdini_patterns))

for principle, patterns in cialdini_patterns.items():
    print(principle, "-", len(patterns), "patterns")

Defined principles: 6
authority - 5 patterns
scarcity - 4 patterns
reciprocity - 4 patterns
commitment_consistency - 5 patterns
liking - 5 patterns
social_proof - 5 patterns


## Feature Extraction Function

The extractor records binary presence, indicator frequency, and optional matched-text evidence for each persuasion principle. Duplicate matches covering the same text span are counted only once.

In [140]:
def extract_cialdini_features(
    text,
    return_matches=False
):
    """Extract Cialdini features from one text value."""

    if pd.isna(text):
        text = ""

    text = str(text)
    features = {}

    for principle, patterns in (
        cialdini_patterns.items()
    ):
        matched_spans = {}

        for pattern in patterns:
            for match in re.finditer(
                pattern,
                text,
                flags=re.IGNORECASE
            ):
                span = match.span()

                if span not in matched_spans:
                    matched_spans[span] = (
                        match.group(0)
                    )

        matched_phrases = [
            matched_spans[span]
            for span in sorted(matched_spans)
        ]

        features[
            f"{principle}_present"
        ] = int(len(matched_phrases) > 0)

        features[
            f"{principle}_count"
        ] = len(matched_phrases)

        if return_matches:
            features[
                f"{principle}_matches"
            ] = matched_phrases

    return features

In [141]:
test_message = (
    "As the program director, I was impressed by "
    "your research. Only a few positions remain, "
    "and I would be happy to help with your application."
)

display(
    pd.Series(
        extract_cialdini_features(
            test_message,
            return_matches=True
        )
    )
)

authority_present                                          1
authority_count                                            1
authority_matches                  [As the program director]
scarcity_present                                           1
scarcity_count                                             1
scarcity_matches                                [Only a few]
reciprocity_present                                        1
reciprocity_count                                          1
reciprocity_matches               [I would be happy to help]
commitment_consistency_present                             0
commitment_consistency_count                               0
commitment_consistency_matches                            []
liking_present                                             1
liking_count                                               1
liking_matches                                [impressed by]
social_proof_present                                       0
social_proof_count      

## Representative Sample Validation

The extractor is applied to the reproducible message sample with matched-text evidence retained for manual review. The evidence is used to evaluate whether each detected feature agrees with its operational definition.

In [142]:
sample_validation_features = (
    message_sample["message"]
    .apply(
        lambda text: extract_cialdini_features(
            text,
            return_matches=True
        )
    )
    .apply(pd.Series)
)

sample_validation_df = pd.concat(
    [
        message_sample[
            [
                "conversation_id",
                "turn_number",
                "scenario_class",
                "label",
                "message"
            ]
        ].reset_index(drop=True),

        sample_validation_features.reset_index(
            drop=True
        )
    ],
    axis=1
)

match_columns = [
    column
    for column in sample_validation_df.columns
    if column.endswith("_matches")
]

display(
    sample_validation_df[
        [
            "conversation_id",
            "turn_number",
            *match_columns
        ]
    ]
)

,conversation_id,turn_number,authority_matches,scarcity_matches,reciprocity_matches,commitment_consistency_matches,liking_matches,social_proof_matches
0,test_0605,11,[],[],[],[],[appreciate your],[]
1,test_0705,10,[],[],[],[],[],[]
2,test_0643,3,[],[],[],[],"[pleasure to connect, impressed by]",[]
3,train_0099,6,[],[],[],[],[appreciate your],[]
4,test_0783,11,[],[],[],[],[],[]
5,test_0883,3,[],[],[],[proceed by],"[glad to hear, impressed with]",[]
6,test_0186,7,[],[time-sensitive],[],[],[],[]
7,test_0387,9,[],[],[],[your continued commitment],[appreciate your],[]
8,test_0081,7,[],[],[],[],[grateful for your],[]
9,train_0323,10,[],[],[],[],[],[]


## Extract Current-Message Features

The validated extractor is applied to all turn-level messages. Labels and scenario fields remain outside the extraction function and do not influence feature values.

In [143]:
current_message_features = (
    turn_df["message"]
    .apply(extract_cialdini_features)
    .apply(pd.Series)
)

feature_turn_df = pd.concat(
    [
        turn_df.reset_index(drop=True),
        current_message_features.reset_index(
            drop=True
        )
    ],
    axis=1
)

print(
    "Feature rows:",
    len(feature_turn_df)
)

print(
    "Feature columns created:",
    len(current_message_features.columns)
)

presence_columns = [
    column
    for column in current_message_features.columns
    if column.endswith("_present")
]

presence_summary = (
    feature_turn_df[presence_columns]
    .sum()
    .rename("Messages detected")
    .to_frame()
)

display(presence_summary)

Feature rows: 14850
Feature columns created: 12


,Messages detected
authority_present,453
scarcity_present,218
reciprocity_present,306
commitment_consistency_present,2783
liking_present,8127
social_proof_present,32


## Stratified Feature Review

A reproducible sample of detected messages is selected separately for each principle. This review checks whether the matched language agrees with the operational definitions before the features are finalized.

In [144]:
review_rows = []

for principle in cialdini_patterns:
    presence_column = (
        f"{principle}_present"
    )

    detected_rows = feature_turn_df.loc[
        feature_turn_df[presence_column] == 1
    ]

    review_sample = detected_rows.sample(
        n=min(3, len(detected_rows)),
        random_state=42
    )

    for _, row in review_sample.iterrows():
        evidence = extract_cialdini_features(
            row["message"],
            return_matches=True
        )

        review_rows.append({
            "principle": principle,
            "conversation_id": row["conversation_id"],
            "turn_number": row["turn_number"],
            "matched_phrases": evidence[
                f"{principle}_matches"
            ],
            "message": row["message"]
        })

manual_review_df = pd.DataFrame(
    review_rows
)

display(manual_review_df)

,principle,conversation_id,turn_number,matched_phrases,message
0,authority,train_0160,4,[official],"Dear Alicia,\n\nI hope this message finds you well. I wanted to reach out and discuss an email you recently received from William at the Global Research Council. While it's exciting to be considered for a grant, it's important to exercise caution when sharing sensitive financial information.\n\n..."
1,authority,test_0217,1,[on behalf of],"Dear Andre, I am reaching out to you on behalf of the prestigious research organization, the International Institute for Advanced Studies (IIAS). Our organization is dedicated to supporting cutting-edge research in the field of renewable energy technologies. Given your impressive background in e..."
2,authority,train_0197,1,[on behalf of],"Dear Claudia,\n\nI hope this message finds you well. I am writing to you on behalf of the prestigious research organization, The Science and Technology Innovation Council (STIC), which is dedicated to advancing knowledge and innovation in the field of renewable energy. I came across your impress..."
3,scarcity,test_0435,5,[Please provide the requested information as soon as possible],"Dear Joseph,\n\nThank you for your prompt and detailed response. Your estimated budget of $25,000 for the next year of research seems reasonable, given your current annual income and lack of other funding sources.\n\nTo further evaluate your eligibility for our funding program, we would like to ..."
4,scarcity,train_0423,4,[time-sensitive],"Hello Carmen,\n\nThank you for providing the next steps in the application process. I am happy to provide the requested information and attach a copy of my resume in PDF format.\n\nPlease find my current address and date of birth below:\n\nAddress: 123 Main Street, Anytown, USA 12345\nDate of Bi..."
5,scarcity,test_0822,11,[Please submit your application as soon as possible],"Dear Jackie,\n\nYou are welcome, and I am glad to hear that you are excited about the possibility of receiving the IEF academic grant. We appreciate your commitment to contributing to the field of Computer Science and AI.\n\nPlease submit your application as soon as possible, and we will review ..."
6,reciprocity,test_0823,7,[I can help],"Hello Larry,\n\nI am glad to hear that you are considering our Scholarship Initiative for Future Technologies (SIFT) and that you found my response helpful. I wish you the best of luck with your application and I look forward to potentially working with you.\n\nPlease let me know if you have any..."
7,reciprocity,test_0727,1,[we can support],"Hello Joseph,\n\nI came across your profile and was impressed by your academic achievements and research interests. I am reaching out on behalf of the 'Global Research Initiative', an organization dedicated to supporting and funding groundbreaking research in the field of renewable energy.\n\nAs..."
8,reciprocity,test_0510,8,[we can help],"Hi Jason,\n\nThank you for your kind words and your continued interest in my insights on the renewable energy industry. I'm more than happy to answer your additional questions.\n\n1. GreenTech Innovations has faced several challenges due to the fluctuating nature of government incentives. For in..."
9,commitment_consistency,test_0212,9,[your commitment],"Dear Richard,\n\nThank you for your message and for your patience as we review your application. We appreciate your interest in the grant from the International Research Foundation (IRF) and your commitment to furthering your research in gene editing.\n\nWe will keep you updated on the status of..."


## Cumulative Persuasion Features

Current-message indicator counts are accumulated within each conversation. At every turn, the cumulative features represent only persuasion indicators observed during the current and preceding turns.

In [145]:
feature_turn_df = feature_turn_df.sort_values(
    [
        "conversation_id",
        "turn_number"
    ]
).reset_index(drop=True)

for principle in cialdini_patterns:
    count_column = f"{principle}_count"

    cumulative_count_column = (
        f"{principle}_cumulative_count"
    )

    cumulative_presence_column = (
        f"{principle}_cumulative_present"
    )

    feature_turn_df[
        cumulative_count_column
    ] = (
        feature_turn_df
        .groupby("conversation_id")[
            count_column
        ]
        .cumsum()
    )

    feature_turn_df[
        cumulative_presence_column
    ] = (
        feature_turn_df[
            cumulative_count_column
        ] > 0
    ).astype(int)

print(
    "Rows with cumulative features:",
    len(feature_turn_df)
)

cumulative_columns = [
    column
    for column in feature_turn_df.columns
    if "_cumulative_" in column
]

print(
    "Cumulative columns created:",
    len(cumulative_columns)
)

display(
    feature_turn_df[
        [
            "conversation_id",
            "turn_number",
            *cumulative_columns
        ]
    ].head()
)

Rows with cumulative features: 14850
Cumulative columns created: 12


,conversation_id,turn_number,authority_cumulative_count,authority_cumulative_present,scarcity_cumulative_count,scarcity_cumulative_present,reciprocity_cumulative_count,reciprocity_cumulative_present,commitment_consistency_cumulative_count,commitment_consistency_cumulative_present,liking_cumulative_count,liking_cumulative_present,social_proof_cumulative_count,social_proof_cumulative_present
0,test_0000,1,0,0,0,0,0,0,0,0,1,1,0,0
1,test_0000,2,0,0,0,0,0,0,0,0,1,1,0,0
2,test_0000,3,0,0,0,0,1,1,2,1,2,1,0,0
3,test_0000,4,0,0,0,0,1,1,2,1,3,1,0,0
4,test_0000,5,0,0,0,0,1,1,3,1,3,1,0,0


## Validate Extracted Features

The extracted features are checked for missing values, invalid binary values, negative counts, and cumulative counts that decrease within a conversation.

In [146]:
feature_columns = [
    column
    for column in feature_turn_df.columns
    if (
        column.endswith("_present")
        or column.endswith("_count")
    )
]

print(
    "Missing feature values:",
    feature_turn_df[
        feature_columns
    ].isna().sum().sum()
)

presence_columns = [
    column
    for column in feature_columns
    if column.endswith("_present")
]

invalid_binary_values = 0

for column in presence_columns:
    invalid_binary_values += (
        ~feature_turn_df[column].isin([0, 1])
    ).sum()

print(
    "Invalid binary values:",
    invalid_binary_values
)

count_columns = [
    column
    for column in feature_columns
    if column.endswith("_count")
]

negative_counts = (
    feature_turn_df[count_columns] < 0
).sum().sum()

print(
    "Negative count values:",
    negative_counts
)

decreasing_cumulative_counts = 0

for principle in cialdini_patterns:
    column = (
        f"{principle}_cumulative_count"
    )

    decreases = (
        feature_turn_df
        .groupby("conversation_id")[column]
        .diff()
        .fillna(0)
        < 0
    )

    decreasing_cumulative_counts += (
        decreases.sum()
    )

print(
    "Decreasing cumulative counts:",
    decreasing_cumulative_counts
)

Missing feature values: 0
Invalid binary values: 0
Negative count values: 0
Decreasing cumulative counts: 0


## Save Feature-Enhanced Data

The validated current-message and cumulative Cialdini features are saved with the turn-level dataset. The original processed dataset remains unchanged.

In [147]:
feature_data_path = (
    project_root
    / "data"
    / "processed"
    / "turn_level_cialdini_features.csv"
)

feature_turn_df.to_csv(
    feature_data_path,
    index=False
)

print(
    "Saved feature data to:",
    feature_data_path
)

print(
    "Saved rows:",
    len(feature_turn_df)
)

print(
    "Saved columns:",
    len(feature_turn_df.columns)
)

Saved feature data to: c:\Users\josie\OneDrive\Desktop\School\Capstone\Phishing Detection\MTPD CAPSTONE\data\processed\turn_level_cialdini_features.csv
Saved rows: 14850
Saved columns: 35


In [148]:
saved_feature_df = pd.read_csv(
    feature_data_path
)

print(
    "Reloaded rows:",
    len(saved_feature_df)
)

print(
    "Reloaded conversations:",
    saved_feature_df[
        "conversation_id"
    ].nunique()
)

print(
    "Reloaded feature columns:",
    len([
        column
        for column in saved_feature_df.columns
        if (
            column.endswith("_present")
            or column.endswith("_count")
        )
    ])
)

Reloaded rows: 14850
Reloaded conversations: 1350
Reloaded feature columns: 24


## Feature Extraction Summary

The table summarizes how frequently each persuasion principle was detected in current messages. These counts describe the rule-based feature output and are not model-performance results.

In [149]:
feature_summary_rows = []

for principle in cialdini_patterns:
    present_column = (
        f"{principle}_present"
    )

    count_column = (
        f"{principle}_count"
    )

    detected_messages = int(
        feature_turn_df[
            present_column
        ].sum()
    )

    total_indicators = int(
        feature_turn_df[
            count_column
        ].sum()
    )

    feature_summary_rows.append({
        "principle": principle,
        "messages_detected": detected_messages,
        "percentage_of_messages": round(
            detected_messages
            / len(feature_turn_df)
            * 100,
            2
        ),
        "total_indicators": total_indicators
    })

feature_summary_df = pd.DataFrame(
    feature_summary_rows
)

display(feature_summary_df)

,principle,messages_detected,percentage_of_messages,total_indicators
0,authority,453,3.05,493
1,scarcity,218,1.47,246
2,reciprocity,306,2.06,311
3,commitment_consistency,2783,18.74,3334
4,liking,8127,54.73,8842
5,social_proof,32,0.22,34
